# Flood ground-truth — explore

Three complementary flood layers, loaded, summarized, and merged into **one
harmonized GeoDataFrame** for pairing with the GOES imagery:

1. **groundsource** (`data/raw/groundsource_2026.parquet`) — [Google Research's
   Groundsource dataset](https://zenodo.org/records/18647054): flood events
   **extracted from local news reports** (Gemini-extracted, 80 languages),
   polygons with a start/end date. ~2.65M rows, global.
2. **flood warnings** (`data/flood_warnings/flood_warnings_conus.parquet`) — NWS
   forecaster-issued **Flash Flood (FF)** + **Areal Flood (FA)** *Warning*
   polygons, CONUS 2019–2026, from IEM.
3. **storm events** (`data/storm_events/storm_events_flood.parquet`) — NCEI
   Storm Events Database: human-confirmed **Flash Flood / Flood** occurrences
   (location points, UTC times, impacts).

All three are fetched by `floodlens.download.flood_data` (`all` subcommand). They
differ in nature — issued *prediction* vs. confirmed *occurrence* vs.
news-reported *extent* — so the unified frame carries a **`kind`** column:

| `kind` | from | meaning |
|---|---|---|
| `warning` | IEM warnings | a forecaster predicted flooding here |
| `verified` | storm events | a human-confirmed flood occurred here |
| `report` | groundsource | local news reported flooding here |

In [ ]:
from datetime import date
from pathlib import Path

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", None)

# repo root (works from notebooks/explore/, notebooks/, or the root itself)
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

GROUNDSOURCE = ROOT / "data/raw/groundsource_2026.parquet"
WARNINGS = ROOT / "data/flood_warnings/flood_warnings_conus.parquet"
STORM_EVENTS = ROOT / "data/storm_events/storm_events_flood.parquet"

# CONUS lon/lat box + study window (matches the warnings coverage)
CONUS = (-125, -66, 24, 49)
START = "2019-01-01"

## 1. Load

In [ ]:
# --- groundsource: observed flood extents ---
gs = gpd.read_parquet(
    GROUNDSOURCE,
    columns=["uuid", "area_km2", "start_date", "end_date", "geometry"],
)
gs["start_date"] = pd.to_datetime(gs["start_date"])
gs["end_date"] = pd.to_datetime(gs["end_date"])

# restrict to the CONUS + 2019-on study window (aligns with the warnings)
gs = gs.cx[CONUS[0]:CONUS[1], CONUS[2]:CONUS[3]]
gs = gs[gs["start_date"] >= START].copy()
print(f"groundsource (CONUS, >= {START}): {len(gs):,} rows")
gs.head(3)

In [ ]:
# --- NWS flood warnings (already CONUS 2019-2026) ---
fw = gpd.read_parquet(WARNINGS)
# IEM timestamps are ISO 'Z' (UTC); make them tz-naive UTC to match groundsource
fw["issue"] = pd.to_datetime(fw["issue"], utc=True).dt.tz_localize(None)
fw["expire"] = pd.to_datetime(fw["expire"], utc=True).dt.tz_localize(None)
print(f"flood warnings: {len(fw):,} rows")
fw.head(3)

In [ ]:
# --- NCEI storm events: confirmed flood occurrences (points) ---
# keep the FF/FA-like flood records (drop Heavy Rain / Debris Flow here).
# points per event = locations table + BEGIN (point_index 0) + END (-1)
se = gpd.read_parquet(STORM_EVENTS)
se = se[se["event_type"].isin(["Flash Flood", "Flood"])].copy()
se["event_id"] = se["event_id"].astype(str)
print(f"storm events (Flash Flood + Flood, CONUS): "
      f"{se['event_id'].nunique():,} events / {len(se):,} points "
      f"({se.geom_source.value_counts().to_dict()})")
se.head(3)

## 2. Summarize

In [ ]:
def summarize(gdf, name, id_col, start_col, end_col, area_col=None):
    """One-line-per-stat summary of a flood GeoDataFrame."""
    print(f"=== {name} ===")
    print(f"rows            : {len(gdf):,}")
    print(f"unique ids      : {gdf[id_col].nunique():,}")
    print(f"date range      : {gdf[start_col].min():%Y-%m-%d} "
          f"-> {gdf[end_col].max():%Y-%m-%d}")
    dur = (gdf[end_col] - gdf[start_col]).dt.days
    print(f"duration (days) : median {dur.median():.0f}, mean {dur.mean():.1f}, "
          f"max {dur.max():.0f}")
    if area_col is not None:
        a = gdf[area_col]
        print(f"area_km2        : sum {a.sum():,.0f}, median {a.median():.2f}, "
              f"mean {a.mean():.1f}, max {a.max():,.0f}")
    print(f"geometry types  : {gdf.geometry.geom_type.value_counts().to_dict()}")
    print(f"bounds (lon/lat): {gdf.total_bounds.round(2).tolist()}")
    print()


summarize(gs, "groundsource (observed extents)", "uuid",
          "start_date", "end_date", "area_km2")
summarize(fw, "NWS flood warnings", "key", "issue", "expire", "area_iem")
# the raw storm-events layer is one POINT per (event x location/begin/end);
# section 3 collapses each event's points into one convex-hull POLYGON
summarize(se, "NCEI storm events (FF + Flood; raw per-point rows)",
          "event_id", "begin_utc", "end_utc")

In [ ]:
# warnings: breakdown by phenomena, year, and polygon source
print("by phenomena:", fw.phenomena.value_counts().to_dict())
print("polygon source:", fw.polygon_source.value_counts().to_dict())
fw.assign(year=fw.issue.dt.year).groupby(["year", "phenomena"]).size().unstack(fill_value=0)

In [ ]:
# groundsource: events per year (by start_date)
gs.assign(year=gs.start_date.dt.year).groupby("year").size()

In [ ]:
# storm events: per year x type, plus impacts and how they were reported
ev = se.drop_duplicates("event_id")
print("flood_cause:", ev.flood_cause.value_counts().head(3).to_dict())
print("top report sources:", ev.report_source.value_counts().head(5).to_dict())
print(f"impacts: ${ev.damage_property_usd.sum() / 1e9:.1f}B property damage, "
      f"{int(ev.deaths_direct.sum()):,} direct deaths")
(ev.assign(year=ev.begin_utc.dt.year)
   .groupby(["year", "event_type"]).size().unstack(fill_value=0))

## 3. Harmonized unified frame

One schema across **all three sources** so they can be filtered, joined to GOES
dates, and overlaid together — one parquet with everything we currently use.
`kind` says what a row *means*; `source` keeps the granular origin. Storm-event
points (locations table + begin/end coordinates, which delineate each event's
spatial span) are collapsed to **one convex-hull polygon per event** — so every
row in the frame is a polygon and `event_id` stays unique.

| column | groundsource | flood warning | storm event |
|---|---|---|---|
| `kind` | `report` | `warning` | `verified` |
| `source` | `groundsource` | `ff_warning` / `fa_warning` | `storm_event` |
| `event_id` | `uuid` | `key` (`year_wfo_ph_etn`) | `EVENT_ID` |
| `phenomena` | `OBS` | `FF` / `FA` | `FF` (Flash Flood) / `FA` (Flood) |
| `issue_date` | `start_date` | `issue` | `begin_utc` |
| `expire_date` | `end_date` | `expire` | `end_utc` |
| `area_km2` | `area_km2` | `area_iem` | hull area (equal-area CRS) |
| `flood_cause` | — | — | NCEI cause |
| `damage_property_usd` | — | — | NCEI damage |
| `geometry` | polygon | polygon | convex hull of the event's points |

In [ ]:
COLS = ["kind", "source", "event_id", "phenomena", "issue_date", "expire_date",
        "area_km2", "flood_cause", "damage_property_usd", "geometry"]

gs_u = gpd.GeoDataFrame({
    "kind": "report",
    "source": "groundsource",
    "event_id": gs["uuid"].astype(str),
    "phenomena": "OBS",
    "issue_date": gs["start_date"],
    "expire_date": gs["end_date"],
    "area_km2": gs["area_km2"],
    "flood_cause": None,
    "damage_property_usd": float("nan"),
    "geometry": gs.geometry,
}, crs=gs.crs)[COLS]

fw_u = gpd.GeoDataFrame({
    "kind": "warning",
    "source": fw["phenomena"].map({"FF": "ff_warning", "FA": "fa_warning"}),
    "event_id": fw["key"],
    "phenomena": fw["phenomena"],
    "issue_date": fw["issue"],
    "expire_date": fw["expire"],
    "area_km2": fw["area_iem"],
    "flood_cause": None,
    "damage_property_usd": float("nan"),
    "geometry": fw.geometry,
}, crs=fw.crs)[COLS]

# storm events: one POLYGON per event — the convex hull of all its points
# (locations + begin/end, which delineate the event's spatial span). Hulls and
# buffers are computed in the equal-area CRS (metres); events whose points are
# collinear or a single point get a 1 km buffer so every row is a polygon.
se_ev = se.dissolve(
    by="event_id",
    aggfunc={"event_type": "first", "begin_utc": "min", "end_utc": "max",
             "flood_cause": "first", "damage_property_usd": "first"},
).reset_index()
hull = se_ev.geometry.to_crs(5070).convex_hull
degenerate = hull.geom_type != "Polygon"
hull[degenerate] = hull[degenerate].buffer(1_000)       # 1 km, in metres
se_area = hull.area / 1e6                               # km^2 (equal-area CRS)
se_ev = se_ev.set_geometry(hull.to_crs(4326))
print(f"storm-event hulls: {len(se_ev):,} polygons "
      f"({int(degenerate.sum()):,} buffered from point/line); "
      f"median {se_area.median():.1f} km2, max {se_area.max():,.0f} km2")

se_u = gpd.GeoDataFrame({
    "kind": "verified",
    "source": "storm_event",
    "event_id": se_ev["event_id"],
    "phenomena": se_ev["event_type"].map({"Flash Flood": "FF", "Flood": "FA"}),
    "issue_date": se_ev["begin_utc"],
    "expire_date": se_ev["end_utc"],
    "area_km2": se_area,
    "flood_cause": se_ev["flood_cause"],
    "damage_property_usd": se_ev["damage_property_usd"],
    "geometry": se_ev.geometry,
}, crs="EPSG:4326")[COLS]

floods = gpd.GeoDataFrame(
    pd.concat([gs_u, fw_u, se_u], ignore_index=True), crs="EPSG:4326"
)
floods["issue_day"] = floods["issue_date"].dt.normalize()   # date key for GOES join
print(f"unified: {len(floods):,} rows")
print("by kind  :", floods.kind.value_counts().to_dict())
print("by source:", floods.source.value_counts().to_dict())
floods.head(3)

### Persist the unified frame

Save to `data/flood_warnings/floods_unified.parquet` — the single parquet other
notebooks load (e.g. `goes_vs_floods.ipynb`). Existing consumers filter by
`source`, so the added `storm_event` rows and the new `kind` /
`flood_cause` / `damage_property_usd` columns are backward-compatible.

In [ ]:
UNIFIED_OUT = ROOT / "data/flood_warnings/floods_unified.parquet"
floods.to_parquet(UNIFIED_OUT)
print(f"wrote {UNIFIED_OUT}  ({len(floods):,} rows)")

In [ ]:
# sanity: schema, null geometry, date span
print(floods.dtypes)
print("\nnull/empty geometry:", int(floods.geometry.isna().sum()),
      "/", int(floods.geometry.is_empty.sum()))
print("issue_date span:", floods.issue_date.min(), "->", floods.issue_date.max())
floods.describe(include="all").T

### Optional: overlay a sample on a map

`.explore()` puts a sample of all three layers on one interactive map, coloured
by `kind` (warning polygons / verified occurrence points / news-report extents).

In [ ]:
floods = floods[floods['issue_date'] >= pd.Timestamp('2026-06-01')]

In [ ]:
floods

In [ ]:
floods.to_parquet(ROOT / "StormArthurEvaluation/data/flood_warnings_2026.parquet")

In [ ]:
sample = floods.sample(1000, random_state=900)
sample.explore(
    column="kind",
    tooltip=["kind", "source", "phenomena", "issue_date", "expire_date",
             "area_km2", "flood_cause"],
    popup=True, cmap="viridis", style_kwds={"fillOpacity": 0.4},
)

## 4. Data inventory

One table of everything assembled so far: the two ground-truth layers loaded
above, plus what's on `/mnt/disk4` from the satellite downloaders (GOES ABI
images from `floodlens.download.goes`, GLM flash day-parquets from
`floodlens.download.glm`).

In [ ]:
# ---------------------------------------------------------------------------
# Data inventory — ground truth (this notebook) vs. satellite data on disk.
# ---------------------------------------------------------------------------
GOES_DIR = Path("/mnt/disk4/goes-data")
GLM_DIR = Path("/mnt/disk4/glm-data")
GLM_RANGE = (date(2019, 1, 1), date(2026, 2, 28))   # glm.py build range

goes_files = list(GOES_DIR.glob("GOES*/*/*/*/*.nc"))
goes_days = {p.parent for p in goes_files}
glm_built = len(list(GLM_DIR.glob("*/glm_flashes_*.parquet")))
glm_total = (GLM_RANGE[1] - GLM_RANGE[0]).days + 1

inventory = pd.DataFrame([
    ("NWS flood warnings (FF + FA)", len(fw),
     f"kind=warning; by phenomena: {fw.phenomena.value_counts().to_dict()}"),
    ("NCEI storm events (FF + Flood)", se["event_id"].nunique(),
     f"kind=verified; confirmed occurrences, {len(se):,} location points"),
    ("groundsource flood events", len(gs),
     f"kind=report; news-reported extents (Google Research), CONUS, >= {START}"),
    ("GOES images downloaded", len(goes_files),
     f"{len(goes_days):,} days x 8 frames/day (full UTC day, 00-21 UTC)"),
    ("GLM flash days", glm_total,
     f"{glm_built:,} day-parquets built ({glm_built / glm_total:.0%}); "
     "~4,320 raw files/day -> 1 parquet/day"),
], columns=["dataset", "count", "detail"])
inventory.style.format({"count": "{:,}"}).hide(axis="index")

## 5. Do NWS warnings verify against groundsource?

Treat each **warning as a prediction** and groundsource as the **observation**:
for every warning, are there groundsource reports **spatially inside the warning
polygon** whose dates overlap the warning's active window? This matters for
training: it tells us how much the two candidate label sources agree, and which
to trust for what.

Matching rule: polygon **intersects** + date overlap of
`[start_date, end_date]` with `[issue_day, expire + 2 days]` — groundsource
extents are day-resolution and a flood's `start_date` can lag the rain, so a
2-day reporting-lag tolerance is allowed past expiry.

In [ ]:
# ---------------------------------------------------------------------------
# Match every warning with the groundsource obs inside it (space AND time).
# ---------------------------------------------------------------------------
LAG = pd.Timedelta(days=2)          # reporting-lag tolerance past warning expiry

# spatial: every (obs, warning) polygon intersection (STRtree-indexed)
pairs = gpd.sjoin(
    gs[["uuid", "start_date", "end_date", "geometry"]],
    fw[["key", "phenomena", "issue", "expire", "geometry"]],
    predicate="intersects", how="inner",
)
# temporal: [start_date, end_date] overlaps [issue_day, expire + LAG]
pairs = pairs[(pairs["start_date"] <= pairs["expire"] + LAG)
              & (pairs["end_date"] >= pairs["issue"].dt.normalize())]
print(f"{len(pairs):,} (obs, warning) matches | "
      f"{pairs['key'].nunique():,} warnings verified by "
      f"{pairs['uuid'].nunique():,} distinct obs")

# per-warning: how many obs landed inside it?
warn_v = fw[["key", "phenomena", "issue", "area_iem", "states"]].copy()
warn_v["n_obs"] = warn_v["key"].map(pairs.groupby("key").size()).fillna(0).astype(int)
warn_v["verified"] = warn_v["n_obs"] > 0
warn_v.sort_values("n_obs", ascending=False).head(8)

In [ ]:
# ---------------------------------------------------------------------------
# Event-level classification report.
#   precision = warnings with >=1 obs inside (verification rate)
#   recall    = obs covered by >=1 warning of that type (detection rate)
# (different units on the two sides -> f1 is indicative, not strict)
# ---------------------------------------------------------------------------
def event_report(phenomena=None):
    w = warn_v if phenomena is None else warn_v[warn_v["phenomena"] == phenomena]
    p = pairs if phenomena is None else pairs[pairs["phenomena"] == phenomena]
    precision = w["verified"].mean()
    recall = p["uuid"].nunique() / len(gs)
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0.0)
    return {"warnings": len(w), "verified": int(w["verified"].sum()),
            "precision (verif. rate)": precision,
            "obs covered": p["uuid"].nunique(), "obs total": len(gs),
            "recall (detection rate)": recall, "f1": f1}


report = pd.DataFrame({"FF (flash flood)": event_report("FF"),
                       "FA (areal flood)": event_report("FA"),
                       "any warning": event_report()}).T
display(report.style.format({"precision (verif. rate)": "{:.1%}",
                             "recall (detection rate)": "{:.1%}", "f1": "{:.2f}",
                             "warnings": "{:,}", "verified": "{:,}",
                             "obs covered": "{:,}", "obs total": "{:,}"}))

# verification rate by year — is agreement stable over time?
(warn_v.assign(year=warn_v["issue"].dt.year)
       .pivot_table(index="year", columns="phenomena", values="verified",
                    aggfunc="mean").round(3))

## 6. Browse all three layers by date

Pick a **date or date range** with the pickers below and hit **draw map**: every
layer active in that window goes on one map as a toggleable overlay (layer
control, top-right) —

- 🔴 **NWS warnings** (predictions) — red outlines
- 🟣 **storm events** (confirmed occurrences) — purple hull polygons
- 🔵 **groundsource** (news-report extents) — blue fills

"Active" = the row's `[issue_date, expire_date]` overlaps the picked window.
Layers with more than a few thousand shapes are sampled to keep the map
responsive (the layer name shows the true count). Defaults to Hurricane Ida
(2021-09-01 → 09-02).

In [ ]:
# ---------------------------------------------------------------------------
# Interactive browser — date / date-range pickers -> all three layers as
# toggleable folium overlays. Uses the unified frame built in section 3.
# ---------------------------------------------------------------------------
import folium
import ipywidgets as widgets
from IPython.display import display

KIND_STYLE = {
    "warning":  {"label": "NWS warnings (predictions)", "color": "#e31a1c",
                 "fillColor": "#e31a1c", "fillOpacity": 0.10, "weight": 1.2},
    "verified": {"label": "storm events (confirmed occurrences)",
                 "color": "#6a3d9a", "fillColor": "#6a3d9a",
                 "fillOpacity": 0.45, "weight": 1.0},
    "report":   {"label": "groundsource (news-report extents)",
                 "color": "#0b4dd6", "fillColor": "#1f78ff",
                 "fillOpacity": 0.45, "weight": 0.5},
}


def flood_map(start, end=None, max_per_layer=4000):
    """All three layers active in [start, end] (inclusive), each toggleable."""
    t0 = pd.Timestamp(start)
    t1 = (pd.Timestamp(end) if end else t0) + pd.Timedelta(days=1)
    act = floods[(floods["issue_date"] < t1) & (floods["expire_date"] >= t0)]
    m = folium.Map(location=[38.5, -96.0], zoom_start=4,
                   tiles="CartoDB positron")
    for kind, style in KIND_STYLE.items():
        sub = act[act["kind"] == kind]
        n = len(sub)
        if not n:
            continue
        note = ""
        if n > max_per_layer:
            sub = sub.sample(max_per_layer, random_state=0)
            note = f", showing {max_per_layer:,}"
        gj = sub[["source", "phenomena", "issue_date", "expire_date",
                  "area_km2", "flood_cause", "geometry"]].copy()
        gj["issue_date"] = gj["issue_date"].astype(str)
        gj["expire_date"] = gj["expire_date"].astype(str)
        gj["flood_cause"] = gj["flood_cause"].fillna("")
        gj["area_km2"] = gj["area_km2"].round(1)
        sf = {k: style[k] for k in ("color", "fillColor", "fillOpacity",
                                    "weight")}
        folium.GeoJson(
            gj.to_json(), name=f"{style['label']} ({n:,}{note})",
            style_function=lambda _f, sf=sf: sf,
            tooltip=folium.GeoJsonTooltip(
                fields=["source", "phenomena", "issue_date", "expire_date",
                        "area_km2", "flood_cause"],
                aliases=["source", "type", "from", "to", "km2", "cause"]),
        ).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


start_w = widgets.DatePicker(description="start", value=date(2021, 9, 1))
end_w = widgets.DatePicker(description="end", value=date(2021, 9, 2))
draw_w = widgets.Button(description="draw map", button_style="primary")
out_w = widgets.Output()


def _draw(_=None):
    with out_w:
        out_w.clear_output()
        display(flood_map(start_w.value, end_w.value))


draw_w.on_click(_draw)
display(widgets.HBox([start_w, end_w, draw_w]), out_w)
_draw()

## 7. Grid-cell classification report — NWS warnings vs observed floods

Section 5 scored warnings *event-by-event*. Here we score them the way the model
is scored: on the **50 km CONUS-land grid** (`config.build_grid_cells`, the same
grid the trainers use), over every **(land cell × day)** in the study window.

- **`y_true`** — the cell-day is **flooded**. Always includes NCEI storm-event
  points; set `INCLUDE_GROUNDSOURCE` to fold in groundsource news-report extents
  too (`True` = the observed union the model targets; `False` = storm events
  only, the single cleanest confirmed-occurrence source).
- **`y_pred`** — an NWS **FF/FA warning** was active over that cell-day
  (`[issue, expire + 2-day lag]`, same lag tolerance as section 5).

Days are **CDT (UTC−5)**, matching the model pipeline: the UTC layers (storm
events, warnings) are shifted −5 h before day-binning so a warning issued the
evening before a flood's reported date is credited to the right day; groundsource
is already day-resolution and used as-is (see `03_vs_nws_warnings.ipynb`). Each
source's active days are rasterized to the cells it intersects (capped at 10 days
per event to keep long open-ended records from smearing). The universe is the
full cell-day matrix, so the `flood` row's support is the observed base rate.

In [ ]:
# ---------------------------------------------------------------------------
# Rasterize each layer to (cell, day) hits on the 50 km CONUS-land grid, then
# score NWS warnings as a binary classifier of observed flooded cell-days.
#   observed floods -> y_true ;  FF/FA warnings -> y_pred
# Day D is CDT (UTC-5), matching the model pipeline: the UTC layers (storm
# events, warnings) are shifted -5 h before binning to days; groundsource is
# already day-resolution and used as-is (see flood-target-cdt-framing).
# (uses the untouched gs / fw / se from section 1 — `floods` was filtered above.)
# ---------------------------------------------------------------------------
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

from floodlens.config import build_grid_cells

INCLUDE_GROUNDSOURCE = True    # y_true = storm events (+ groundsource news reports if True)
CDT = pd.Timedelta(hours=5)    # day D is CDT (UTC-5): shift UTC layers before day-binning

cells, GRID_R, GRID_C, _land = build_grid_cells()      # EPSG:4326, cols R/C/cell_id
DAY_MAX = pd.Timestamp("2025-12-31")                   # full-label span (2026 labels truncated)
DAYS = pd.date_range(START, DAY_MAX, freq="D")
CAP_DAYS = 10                                          # per-event day cap (open-ended records)
print(f"grid {GRID_R}x{GRID_C} | {len(cells):,} land cells x {len(DAYS):,} days "
      f"= {len(cells) * len(DAYS):,} cell-days")


def cell_day_hits(gdf, start_col, end_col):
    """Set of (cell_id, day) a set of dated geometries touches (space AND time)."""
    j = gpd.sjoin(cells[["cell_id", "geometry"]], gdf, predicate="intersects",
                  how="inner")
    lo, hi = DAYS[0], DAYS[-1]
    hits = set()
    for cid, s, e in zip(j["cell_id"], j[start_col], j[end_col]):
        s, e = pd.Timestamp(s).normalize(), pd.Timestamp(e).normalize()
        if pd.isna(s) or pd.isna(e):
            continue
        days = pd.date_range(max(s, lo), min(e, hi), freq="D")
        for d in days[:CAP_DAYS]:
            hits.add((cid, d))
    return hits


# storm events: UTC -> CDT before day-binning
se_cdt = se[["begin_utc", "end_utc", "geometry"]].assign(
    begin_cdt=se["begin_utc"] - CDT, end_cdt=se["end_utc"] - CDT)
se_hits = cell_day_hits(se_cdt, "begin_cdt", "end_cdt")

# groundsource: already day-resolution, used as-is (no CDT shift)
gs_hits = (cell_day_hits(gs[["start_date", "end_date", "geometry"]],
                         "start_date", "end_date")
           if INCLUDE_GROUNDSOURCE else set())

# warnings: UTC -> CDT, then extend expiry by the reporting-lag tolerance (LAG, section 5)
fw_cdt = fw[["issue", "expire", "phenomena", "geometry"]].assign(
    issue_cdt=fw["issue"] - CDT, expire_cdt=fw["expire"] - CDT + LAG)
warn_hits = cell_day_hits(fw_cdt, "issue_cdt", "expire_cdt")

obs_hits = se_hits | gs_hits                           # observed flood = target
src = "storm events + groundsource" if INCLUDE_GROUNDSOURCE else "storm events only"
print(f"target = {src}  (CDT day binning)")
print(f"observed cell-days: {len(obs_hits):,} "
      f"(storm {len(se_hits):,} | groundsource {len(gs_hits):,}) | "
      f"warned cell-days: {len(warn_hits):,}")

# --- flatten to y_true / y_pred over the full (cell x day) universe ---
cell_pos = {c: i for i, c in enumerate(cells["cell_id"].to_numpy())}
day_pos = {d: i for i, d in enumerate(DAYS)}
n = len(cells) * len(DAYS)


def to_flat(hits):
    out = np.zeros(n, dtype=np.int8)
    idx = [cell_pos[c] * len(DAYS) + day_pos[d] for c, d in hits if d in day_pos]
    out[idx] = 1
    return out


y_true = to_flat(obs_hits)
y_pred = to_flat(warn_hits)
print(f"\nobserved base rate: {y_true.mean():.4%}  ({y_true.sum():,} flooded cell-days)\n")

print(f"NWS warnings as a predictor of observed flooded cell-days "
      f"(50 km grid; target = {src}):\n")
print(classification_report(y_true, y_pred, target_names=["no flood", "flood"],
                            digits=3))
cm = confusion_matrix(y_true, y_pred)
display(pd.DataFrame(cm, index=["observed: dry", "observed: flood"],
                     columns=["no warning", "warning"]))